In [ ]:
# ============================================================
# STEP 6: UrduSpeech subset + benchmark + FLEURS merge (Colab)
# ============================================================
import os, sys, random, shutil
from collections import Counter

# ---------- 0. Repo, libraries, optional HF token ----------
REPO_URL = "https://github.com/FatimaaShaheen/urdu-asr-subtitles.git"
REPO_DIR = "/content/urdu-asr-subtitles"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

!pip install -q "datasets<4" huggingface_hub soundfile librosa jiwer

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF token mil gaya (secrets se)")
except Exception:
    print("HF token nahi laga (theek hai, bina token bhi chalega)")

import matplotlib.pyplot as plt
from IPython.display import Audio, display
from datasets import concatenate_datasets

from src.data.extra_datasets import (
    list_repo, categories, select, download, build_dataset,
    fleurs_to_common, CORPUS_STD, BENCH_STD,
)
from src.data.filters import split_speakers
from src.data.load_dataset import load_fleurs_urdu
from src.data.clean_text import normalize_for_wer

# ---------- Settings ----------
TRAIN_CATS = ["news", "interview", "podcast", "vlogs", "roadside_interview"]
N_PER_CAT  = 600      # corpus: har category se kitni clips
N_BENCH    = 100      # benchmark: har category se kitni clips
RAW   = "/content/us_raw"
CLEAN = "/content/us_16k"

# ---------- 1. Categories dekho ----------
files = list_repo()
print("Corpus short categories:", categories(files, CORPUS_STD, "short"))
bench_cats = categories(files, BENCH_STD, "short")
print("Benchmark short categories:", bench_cats)
missing = [c for c in TRAIN_CATS if c not in categories(files, CORPUS_STD, "short")]
assert not missing, f"Yeh categories corpus mein nahi: {missing}"

# ---------- 2. Download ----------
train_paths = select(files, CORPUS_STD, "short", TRAIN_CATS, N_PER_CAT)
bench_paths = select(files, BENCH_STD, "short", bench_cats, N_BENCH)
print(f"\nDownload: corpus {len(train_paths)} files, benchmark {len(bench_paths)} files")
download(train_paths, f"{RAW}/corpus")
download(bench_paths, f"{RAW}/bench")

# ---------- 3. Filter + 16kHz mono + Dataset ----------
urdu, st1  = build_dataset(f"{RAW}/corpus", train_paths, CLEAN, "urduspeech")
bench, st2 = build_dataset(f"{RAW}/bench",  bench_paths, CLEAN, "urduspeech_bench")
print("\nCorpus subset:", len(urdu), "examples,",
      f"{sum(urdu['duration'])/3600:.2f} ghante, hataye gaye: {st1}")
print("Benchmark:    ", len(bench), "examples,",
      f"{sum(bench['duration'])/3600:.2f} ghante, hataye gaye: {st2}")
print(Counter(urdu["category"]))

# ---------- 4. Haath se check: 3 examples suno aur text parho ----------
for i in random.Random(1).sample(range(len(urdu)), 3):
    ex = urdu[i]
    print(f"\n[{ex['category']}] {ex['text']}")
    display(Audio(ex["audio"]["array"], rate=ex["audio"]["sampling_rate"]))

# ---------- 5. Duration aur bolne ki raftaar ----------
cps = [len(t.replace(" ", "")) / d for t, d in zip(urdu["text"], urdu["duration"])]
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(urdu["duration"], bins=30); ax[0].set_title("Duration (sec)")
ax[1].hist(cps, bins=30);              ax[1].set_title("Characters per second")
plt.show()

# ---------- 6. Speaker-wise split (speaker leakage se bachao) ----------
held = split_speakers(urdu["speaker"], holdout_frac=0.05)
urdu_val   = urdu.filter(lambda s: s in held, input_columns=["speaker"])
urdu_train = urdu.filter(lambda s: s not in held, input_columns=["speaker"])
print(f"\nUrduSpeech train: {len(urdu_train)}, val: {len(urdu_val)}, "
      f"held-out speakers: {len(held)}")

# ---------- 7. FLEURS ko same schema mein lao aur merge karo ----------
f_train = fleurs_to_common(load_fleurs_urdu("train"))
f_val   = fleurs_to_common(load_fleurs_urdu("validation"))
merged = concatenate_datasets([f_train, urdu_train])
merged = merged.filter(lambda d: d <= 30.0, input_columns=["duration"])  # Whisper 30s limit
print("\nMerged train:", len(merged), "examples,",
      f"{sum(merged['duration'])/3600:.2f} ghante")
print(Counter(merged["source"]))

# ---------- 8. Leakage checks ----------
fleurs_test_norm = {normalize_for_wer(t) for t in load_fleurs_urdu("test")["transcription"]}
train_texts = set(merged["text"])
print("\nFLEURS-test ke jumle UrduSpeech train mein :",
      sum(1 for t in urdu["text"] if t in fleurs_test_norm))
print("Benchmark ke jumle merged train mein       :",
      sum(1 for t in bench["text"] if t in train_texts))
print("Benchmark ke jumle FLEURS-test mein        :",
      sum(1 for t in bench["text"] if t in fleurs_test_norm))

# ---------- 9. Kachra saaf aur size dekho ----------
shutil.rmtree(RAW, ignore_errors=True)   # original bari files hata do
!du -sh {CLEAN}

# ---------- 10. Google Drive mein mehfooz karo (Step 7 ke liye) ----------
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/urdu_asr"
merged.save_to_disk(f"{OUT}/merged_train")
f_train.save_to_disk(f"{OUT}/fleurs_train_only")
f_val.save_to_disk(f"{OUT}/fleurs_val")
urdu_val.save_to_disk(f"{OUT}/urduspeech_val")
bench.save_to_disk(f"{OUT}/urduspeech_bench")
print("\nSave ho gaya:", OUT)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
!df -h /content/drive | tail -1

In [ ]:
import os, numpy as np, datasets
from datasets import Audio, Dataset, Features, Value, concatenate_datasets

# ---------- 1. Benchmark se woh jumle hatao jo train mein hain ----------
train_texts = set(merged["text"])
dup = [t for t in bench["text"] if t in train_texts]
print("Overlap jumle:", dup)
bench = bench.filter(lambda t: t not in train_texts, input_columns=["text"])
print("Benchmark ab:", len(bench), "clips")

# ---------- 2. Audio ko dataset ke andar embed karo ----------
FEATS = Features({"audio": Audio(sampling_rate=16000), "text": Value("string"),
                  "duration": Value("float64"), "source": Value("string"),
                  "speaker": Value("string"), "category": Value("string")})

def embed_audio(ds):
    raw = ds.cast_column("audio", Audio(decode=False))
    audio = []
    for a in raw["audio"]:
        b = a["bytes"] if a["bytes"] is not None else open(a["path"], "rb").read()
        audio.append({"bytes": b, "path": os.path.basename(a["path"] or "x.wav")})
    cols = {c: list(ds[c]) for c in FEATS if c != "audio"}
    cols["audio"] = audio
    return Dataset.from_dict(cols, features=FEATS)

urdu_train_e = embed_audio(urdu_train)
urdu_val_e   = embed_audio(urdu_val)
bench_e      = embed_audio(bench)
merged = concatenate_datasets([f_train, urdu_train_e]).filter(
    lambda d: d <= 30.0, input_columns=["duration"])

# ---------- 3. Drive mein save ----------
OUT = "/content/drive/MyDrive/urdu_asr"
merged.save_to_disk(f"{OUT}/merged_train")
f_val.save_to_disk(f"{OUT}/fleurs_val")
urdu_val_e.save_to_disk(f"{OUT}/urduspeech_val")
bench_e.save_to_disk(f"{OUT}/urduspeech_bench")
print("Save ho gaya:", OUT)

# ---------- 4. Verify: asal audio files chhupa kar dobara parho ----------
os.rename("/content/us_16k", "/content/us_16k_bak")
try:
    for n in ["merged_train", "fleurs_val", "urduspeech_val", "urduspeech_bench"]:
        d = datasets.load_from_disk(f"{OUT}/{n}")
        for i in [0, len(d) // 2, len(d) - 1]:     # pehla, beech wala, aakhri (merged mein UrduSpeech)
            _ = d[i]["audio"]["array"]
        print("PASS:", n, len(d), "examples,", f"{sum(d['duration'])/3600:.2f} ghante")
except Exception as e:
    print("FAIL:", repr(e)[:400])
finally:
    os.rename("/content/us_16k_bak", "/content/us_16k")